# Independent acoustic developmental LM — 50-hour log-Mel pilot

This notebook trains the new **independent** causal acoustic path. It does not alter or load the existing IPA-feature Phase 1 checkpoints. Select a T4 GPU before running.

Default mode prepares TalkBank **Providence** adult/caregiver timestamped CHAT tiers directly after a runtime-only TalkBank login. It excludes `CHI`; credentials/cookies are never printed or saved. An upload-ZIP fallback is also available for other properly licensed adult/caregiver child-directed WAV data. See `docs/acoustic_logmel_phase1.md` in the cloned repository for scope and limitations.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git'
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists():
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{PROJECT}[acoustic]', 'pytest'], check=True)
os.chdir(PROJECT)
print('Project ready:', PROJECT)
print('Commit:', subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Optional code-level smoke tests; they use synthetic WAV only and do not train your data.
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_acoustic_phase1.py'], check=True)

## Prepare Providence or upload adult/caregiver child-directed audio

`SOURCE_MODE = 'providence'` is the direct TalkBank option. It prompts for your TalkBank credentials using `getpass`, downloads the official transcript ZIP, downloads only needed linked media, cuts timestamped MOT/FAT/other adult tiers to PCM WAV, and creates the strict manifest. `directed_to_child=true` means parent-child corpus context; CHAT does not encode the addressee of every individual tier.

`SOURCE_MODE = 'upload_zip'` uploads one ZIP containing exactly one `.tsv`/`.csv` manifest and its referenced `.wav` files. Required manifest columns: `audio_path`, `corpus_id`, `session_id`, `target_child_age_months`, `source_corpus`, `speaker_role`, `directed_to_child`; optional: `recording_order`, `split`.

In [ ]:
import shutil, zipfile
from getpass import getpass
from google.colab import files

SOURCE_MODE = 'providence'  # change only to 'upload_zip' for your own adult/CDS WAV + manifest archive
if SOURCE_MODE not in {'providence', 'upload_zip'}:
    raise ValueError('SOURCE_MODE must be providence or upload_zip')
INPUT_ROOT = pathlib.Path('/content/acoustic_cds_input')
if INPUT_ROOT.exists():
    shutil.rmtree(INPUT_ROOT)
INPUT_ROOT.mkdir(parents=True)
if SOURCE_MODE == 'providence':
    # Neither value is printed, saved, written to a config nor included in outputs.
    talkbank_email = input('TalkBank email: ')
    talkbank_password = getpass('TalkBank password: ')
    from devlm.acoustic.providence import prepare_providence
    AUDIO_MANIFEST = prepare_providence(
        INPUT_ROOT, email=talkbank_email, password=talkbank_password,
        train_hours=50.0, validation_hours=0.5, validation_fraction=0.1, seed=20261005,
    )
    del talkbank_email, talkbank_password
else:
    print('Select exactly one ZIP containing the WAV files plus one manifest TSV/CSV.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Expected exactly one ZIP upload; rerun this cell.')
    archive_name, archive_bytes = next(iter(uploaded.items()))
    if not archive_name.lower().endswith('.zip'):
        raise ValueError('Input must be a ZIP archive.')
    archive_path = INPUT_ROOT / pathlib.Path(archive_name).name
    archive_path.write_bytes(archive_bytes)
    with zipfile.ZipFile(archive_path) as archive:
        safe_root = INPUT_ROOT.resolve()
        for member in archive.infolist():
            destination = (INPUT_ROOT / member.filename).resolve()
            try:
                destination.relative_to(safe_root)
            except ValueError as exc:
                raise ValueError(f'Unsafe ZIP member path: {member.filename!r}') from exc
        archive.extractall(INPUT_ROOT)
    manifests = sorted([p for p in INPUT_ROOT.rglob('*') if p.suffix.lower() in {'.tsv', '.csv'}])
    if len(manifests) != 1:
        raise ValueError(f'Expected exactly one TSV/CSV manifest in ZIP; found {len(manifests)}: {manifests}')
    AUDIO_MANIFEST = manifests[0]
print('Manifest:', AUDIO_MANIFEST)

In [ ]:
# Fail-fast data audit: validates adult/CDS metadata and reports available duration.
import wave, torch
from collections import Counter
from devlm.acoustic.data import load_audio_manifest, split_audio_sessions

items = load_audio_manifest(AUDIO_MANIFEST)
def wav_seconds(path):
    with wave.open(str(path), 'rb') as handle:
        return handle.getnframes() / handle.getframerate()
seconds = sum(wav_seconds(item.audio_path) for item in items)
train_items, validation_items = split_audio_sessions(items, validation_fraction=0.1, seed=20261005)
train_seconds = sum(wav_seconds(item.audio_path) for item in train_items)
print(f'Validated {len(items):,} adult/caregiver-to-child WAV recordings, {seconds / 3600:.3f} total raw hours.')
print(f'After the deterministic session split: {train_seconds / 3600:.3f} train hours, {sum(wav_seconds(item.audio_path) for item in validation_items) / 3600:.3f} validation hours.')
print('Source corpora:', dict(Counter(item.source_corpus for item in items)))
print('Speaker roles:', dict(Counter(item.speaker_role for item in items)))
if train_seconds / 3600 < 50:
    print('WARNING: fewer than 50 training-split hours are available. The run will train on all eligible training-split audio, not invent missing exposure.')
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. In Colab select Runtime > Change runtime type > T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json, torch

OUTPUT_DIR = pathlib.Path('/content/devlm_acoustic_logmel_50h_outputs')
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)  # known local Colab output path only; prevents stale checkpoints on rerun
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def toml_string(value):
    return json.dumps(str(value))

config_text = f'''[acoustic_phase1]
audio_manifest_path = {toml_string(AUDIO_MANIFEST)}
output_dir = {toml_string(OUTPUT_DIR)}
seed = 20261005
device = "cuda"
validation_fraction = 0.1
sample_rate = 16000
n_mels = 80
n_fft = 400
hop_length = 160
hidden_size = 128
num_layers = 1
dropout = 0.0
learning_rate = 0.001
gradient_clip_norm = 1.0
sequence_chunk_frames = 4096
future_horizons_frames = [3, 5, 10]
max_train_hours = 50.0
max_validation_hours = 0.5
normalization_max_hours = 1.0
target_checkpoint_count = 30
'''
COLAB_CONFIG = pathlib.Path('/content/acoustic_logmel_50h.toml')
COLAB_CONFIG.write_text(config_text, encoding='utf-8')
print(config_text)

In [ ]:
# Causal 30/50/100-ms future log-Mel prediction; live progress and each checkpoint's validation loss print here.
subprocess.run([sys.executable, '-u', '-m', 'devlm.acoustic.cli', '--config', str(COLAB_CONFIG)], check=True)

In [ ]:
checkpoints = sorted(OUTPUT_DIR.glob('acoustic_checkpoint_step_*.pt'))
if not checkpoints:
    raise FileNotFoundError('No acoustic checkpoints found; finish the training cell first.')
archive_path = shutil.make_archive('/content/devlm_acoustic_logmel_50h_outputs', 'zip', root_dir=OUTPUT_DIR)
print(f'Archived {len(checkpoints)} acoustic checkpoints to {archive_path}')
files.download(archive_path)